In [ ]:
import pandas as pd
from src.features import build_feature_matrix

# Load raw data
eua = pd.read_csv("data/raw/raw_eua.csv", index_col=0, parse_dates=True)
ttf = pd.read_csv("data/raw/raw_ttf.csv", index_col=0, parse_dates=True)
coal = pd.read_csv("data/raw/raw_coal.csv", index_col=0, parse_dates=True)
weather = pd.read_csv("data/raw/raw_weather.csv", index_col=0, parse_dates=True)

# Handle optional generation data if it exists
try:
    gen = pd.read_csv("data/raw/raw_generation.csv", index_col=0, parse_dates=True)
except FileNotFoundError:
    # Fallback empty dataframe with required columns if ENTSO-E was skipped
    gen = pd.DataFrame(index=eua.index)
    gen["renewable_share"] = 0.3  # proxy constant
    gen["fossil_share"] = 0.4

# Build matrix and save to processed
df = build_feature_matrix(eua, ttf, coal, gen, weather, None)
df.to_csv("data/processed/final_feature_matrix.csv")
print("Feature matrix created with shape:", df.shape)

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_report(series, name):
    result = adfuller(series.dropna(), autolag="AIC")
    print(f"--- ADF Test: {name} ---")
    print(f"  p-value: {result[1]:.4f}")
    if result[1] < 0.05:
        print("  → STATIONARY (Good for linear modeling)\n")
    else:
        print("  → NON-STATIONARY (Needs differencing)\n")

df = pd.read_csv("data/processed/final_feature_matrix.csv", index_col=0, parse_dates=True)

# Test price levels vs log returns
adf_report(df["eua_price"], "EUA Price (Levels)")
adf_report(df["d_log_eua"], "Δ Log EUA (Returns)")